In [1]:

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# 1. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName("Ecommerce_ML_Pipeline") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .getOrCreate()

# 2. ĐỌC DỮ LIỆU SẠCH TỪ HDFS
hdfs_cleaned_path = "hdfs://localhost:9000/user/bigdata/data/cleaned_dataset"

# Đọc file CSV đã sạch.
df_cleaned = spark.read.csv(hdfs_cleaned_path, header=True, inferSchema=True)

print(f"--- ĐÃ NẠP DỮ LIỆU SẠCH. SỐ LƯỢNG BẢN GHI: {df_cleaned.count()} ---")
df_cleaned.printSchema() # Kiểm tra lại kiểu dữ liệu

--- ĐÃ NẠP DỮ LIỆU SẠCH. SỐ LƯỢNG BẢN GHI: 1000000 ---
root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: double (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- cart_abandonment_rate: integer (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: integer (nullable = true)
 |-- preferred_payment_meth

In [3]:

# 1. Khai báo các cột cho bài toán Phân cụm (K-Means)
categorical_cols = ["preferred_payment_method"]
numeric_cols = ["monthly_spend", "coupon_usage_frequency", "weekly_purchases", "browse_to_buy_ratio"]

# 2. Khởi tạo các stages cho Pipeline
indexer = StringIndexer(inputCol="preferred_payment_method", outputCol="payment_indexed", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="payment_indexed", outputCol="payment_encoded")

assembler_inputs = ["payment_encoded"] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features", handleInvalid="skip")

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

# 3. Đóng gói và biến đổi dữ liệu
pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler])
pipeline_model = pipeline.fit(df_cleaned)
df_ml_ready = pipeline_model.transform(df_cleaned)



In [7]:
# 4. Trích xuất dữ liệu đầu vào cho Machine Learning
df_model_input = df_ml_ready.select("user_id", "features")

print("--- MẪU DỮ LIỆU ĐÃ QUA PIPELINE (SẴN SÀNG CHO ML) ---")
df_beautiful = df_model_input.limit(5).toPandas()
display(df_beautiful)

--- MẪU DỮ LIỆU ĐÃ QUA PIPELINE (SẴN SÀNG CHO ML) ---


,user_id,features
0,12,"[-0.4477225620016961, -0.44771773391685427, -0..."
1,22,"[-0.4477225620016961, -0.44771773391685427, 2...."
2,26,"[-0.4477225620016961, -0.44771773391685427, -0..."
3,27,"[-0.4477225620016961, -0.44771773391685427, 2...."
4,44,"[-0.4477225620016961, -0.44771773391685427, -0..."


In [8]:

# 1. Chia tập Train/Test khoa học
train_data, test_data = df_model_input.randomSplit([0.8, 0.2], seed=42)

print(f"Số lượng bản ghi tập Train (80%): {train_data.count()}")
print(f"Số lượng bản ghi tập Test (20%): {test_data.count()}")

# 2. Lưu kết quả xuống HDFS bằng định dạng PARQUET
hdfs_train_path = "hdfs://localhost:9000/user/bigdata/data/ml_train_data.parquet"
hdfs_test_path = "hdfs://localhost:9000/user/bigdata/data/ml_test_data.parquet"

train_data.write.parquet(hdfs_train_path, mode="overwrite")
test_data.write.parquet(hdfs_test_path, mode="overwrite")

print("--- HOÀN TẤT LƯU TRỮ DỮ LIỆU HỌC MÁY ---")

Số lượng bản ghi tập Train (80%): 800296
Số lượng bản ghi tập Test (20%): 199704
--- HOÀN TẤT LƯU TRỮ DỮ LIỆU HỌC MÁY ---
